# Fine-tuning CLLIP on ROCOv2 (Medical Imaging)

In [1]:
# setup environment

# CUDA + PyTorch + CLIP
!pip install torch torchvision --quiet
!pip install git+https://github.com/openai/CLIP.git --quiet
!pip install datasets --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from google.colab import drive, files
import torch
import torch.nn.functional as F
import clip
from torch.utils.data import DataLoader
from tqdm import tqdm
import math

In [3]:
# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Load dataset
dataset = load_dataset("eltorio/ROCOv2-radiology")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

data/train-00000-of-00027.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/423M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/59962 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9904 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9927 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [5]:
# Setup device and model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print(f"Using device: {device}")

100%|███████████████████████████████████████| 338M/338M [00:07<00:00, 45.8MiB/s]


Using device: cuda


In [6]:
# Preprocess function
def preprocess_example(example):
    """Preprocess image and caption for CLIP"""
    try:
        # Preprocess image
        example["pixel_values"] = preprocess(example["image"])

        # Tokenize caption with truncation
        tokenized = clip.tokenize([example["caption"]], truncate=True)
        example["input_ids"] = tokenized[0]

        return example
    except Exception as e:
        print(f"Error preprocessing: {e}")
        return None

# Create small training subset
small_train = dataset["train"].shuffle(seed=42).select(range(5000))

# Apply preprocessing
print("Preprocessing dataset...")
small_train = small_train.map(
    preprocess_example,
    remove_columns=["image", "caption"],
    desc="Preprocessing"
)

# Filter out any failed preprocessings
small_train = small_train.filter(lambda x: x is not None)

Preprocessing dataset...


Preprocessing:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [7]:
# Collate function
def collate_fn(batch):
    """Stack batch items into tensors"""
    return {
        "images": torch.stack([b["pixel_values"] for b in batch]).to(device),
        "input_ids": torch.stack([b["input_ids"] for b in batch]).to(device),
    }

# Create DataLoader
train_loader = DataLoader(
    small_train,
    batch_size=32,  # Increased batch size for stability
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

print(f"Training samples: {len(small_train)}")
print(f"Batches per epoch: {len(train_loader)}")

Training samples: 5000
Batches per epoch: 157


In [8]:
# Initialize learnable temperature parameter
logit_scale = model.logit_scale.clone().detach().requires_grad_(True)

# Setup optimizer - train BOTH encoders
optimizer = torch.optim.AdamW(
    [
        {"params": model.visual.parameters(), "lr": 1e-6},  # Lower LR for image encoder
        {"params": model.transformer.parameters(), "lr": 5e-6},  # Text encoder
        {"params": [logit_scale], "lr": 1e-5}  # Temperature
    ],
    weight_decay=0.01
)

# Scheduler for learning rate warmup and decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3 * len(train_loader))

print(f"Initial temperature: {logit_scale.exp().item():.4f}")

Initial temperature: 100.0000


In [9]:
# Fine-tuning loop
num_epochs = 3
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    num_batches = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in enumerate(progress_bar):
        imgs = batch["images"]
        txt = batch["input_ids"]

        # Encode images and text (both trainable now)
        img_features = model.encode_image(imgs)
        txt_features = model.encode_text(txt)

        # Normalize features
        img_features = F.normalize(img_features, dim=-1, p=2)
        txt_features = F.normalize(txt_features, dim=-1, p=2)

        # Clamp logit_scale to prevent explosion
        logit_scale.data = torch.clamp(logit_scale.data, max=math.log(100))

        # Compute similarity logits
        logits = logit_scale.exp() * img_features @ txt_features.T

        # Create labels (diagonal matches)
        labels = torch.arange(len(imgs), device=device)

        # Symmetric contrastive loss
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        # Check for NaN
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"\nWarning: NaN/Inf loss at batch {batch_idx}, skipping...")
            continue

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_([logit_scale], max_norm=1.0)

        optimizer.step()
        scheduler.step()

        # Track loss
        total_loss += loss.item()
        num_batches += 1

        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'temp': f'{logit_scale.exp().item():.2f}',
            'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
        })

    # Epoch summary
    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    print(f"\nEpoch {epoch+1} - Average Loss: {avg_loss:.4f}, Temperature: {logit_scale.exp().item():.4f}")

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'logit_scale': logit_scale,
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, "/content/drive/MyDrive/clip_med_finetuned_best.pt")
        print(f"Saved best model (loss: {avg_loss:.4f})")

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'logit_scale': logit_scale,
}, "/content/drive/MyDrive/clip_med_finetuned_final.pt")

print("\n✓ Training complete!")

Epoch 1/3:   0%|          | 0/157 [00:04<?, ?it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1295337459.py", line 5, in collate_fn
    "images": torch.stack([b["pixel_values"] for b in batch]).to(device),
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: expected Tensor as element 0 in argument 0, but got list


In [ ]:
# Validation/Testing
print("\nTesting on validation example...")
model.eval()

test_example = dataset["validation"][0]
img = preprocess(test_example["image"]).unsqueeze(0).to(device)
captions = ["a chest X-ray", "a PET scan", "an ultrasound image"]
txt = clip.tokenize(captions).to(device)

with torch.no_grad():
    img_feat = model.encode_image(img)
    txt_feat = model.encode_text(txt)
    img_feat = F.normalize(img_feat, dim=-1, p=2)
    txt_feat = F.normalize(txt_feat, dim=-1, p=2)
    scores = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1)

print("\nPrediction scores:")
for cap, score in zip(captions, scores[0]):
    print(f"  {cap}: {score.item():.4f}")

print(f"\nActual caption: {test_example['caption']}")